# Mancino — Current Promotions (Public GitHub URL)

This notebook loads the **public** promotions file directly from GitHub and shows today's active promotions.

In [5]:
# Import Libraries
import pandas as pd
from datetime import datetime
try:
    from zoneinfo import ZoneInfo
    TZ = ZoneInfo("America/New_York")
except Exception:
    TZ = None

In [6]:
# Establishing the path to the CSV file
# Note: Make sure of updating this, if your file path changes

PROMOS_URL = "https://raw.githubusercontent.com/jrmst102/mancino/main/data/v1_2025-09-21/promotions.csv"

In [7]:
# Reading CSV File
print("Reading:", PROMOS_URL)
df = pd.read_csv(PROMOS_URL, dtype=str)

Reading: https://raw.githubusercontent.com/jrmst102/mancino/main/data/v1_2025-09-21/promotions.csv


In [11]:
# Normalizing the dates to ensure they are displayed properly. For the purpose of this project, weeks start on Sundays.
# Also, this notebook shows the current promotions. To do so, it needs to know today's date, and then show all the promotions from the past Sunday, until Saturday


# Normalize date/bool
for c in ("week_start","week_end"):
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce").dt.date

def to_bool(x, default=True):
    if x is None or (isinstance(x,float) and pd.isna(x)):
        return default
    return str(x).strip().lower() in {"true","1","t","yes","y"}

if "active" in df.columns:
    df["active"] = df["active"].map(lambda x: to_bool(x, True))
else:
    df["active"] = True

# Calculate and display today's date

today = (datetime.now(TZ).date() if TZ else pd.Timestamp.today().date())
print("Today:", today)

# Create a new, empty dataframe, called 'current'

current = df[
    df["active"].fillna(True)
    & df["week_start"].notna()
    & df["week_end"].notna()
    & (df["week_start"] <= today)
    & (df["week_end"] >= today)
].copy()


Today: 2025-09-19


In [15]:
# Visualizing the CSV file
# Simple view

from IPython.display import display
if current.empty:
    print("No active promotions for today.")
else:
    keep = [
        "promotion_id","promo_type","name","week_start","week_end",
        "scope_type","scope_id","store_scope",
        "amount_off","percent_off","new_price","buy_qty","get_qty",
        "bundle_qty","bundle_price","min_qty","min_spend",
        "priority","can_stack","notes"
    ]
    present = [c for c in keep if c in current.columns]
    by = [c for c in ["priority","promo_type","store_scope"] if c in present]
    asc = [False] + [True]*(len(by)-1) if by and by[0]=="priority" else [True]*len(by)
    current = current[present].sort_values(by=by, ascending=asc).reset_index(drop=True)
    display(current)


,promotion_id,promo_type,name,week_start,week_end,scope_type,scope_id,store_scope,amount_off,percent_off,new_price,buy_qty,get_qty,bundle_qty,bundle_price,min_qty,min_spend,priority,can_stack,notes
0,PR2025091414,BUNDLE,2 for $11.54,2025-09-14,2025-09-20,category,Pet Food,2,NaN,NaN,NaN,NaN,NaN,2,11.54,NaN,NaN,90,FALSE,select set
1,PR2025091401,BUNDLE,3 for $5.81,2025-09-14,2025-09-20,category,Candy,3,NaN,NaN,NaN,NaN,NaN,3,5.81,NaN,NaN,90,FALSE,mix and match allowed
2,PR2025091403,BUNDLE,3 for $8.87,2025-09-14,2025-09-20,category,Baking,ALL,NaN,NaN,NaN,NaN,NaN,3,8.87,NaN,NaN,90,FALSE,select set
3,PR2025091408,BUNDLE,3 for $16.55,2025-09-14,2025-09-20,category,Prepared Meals,ALL,NaN,NaN,NaN,NaN,NaN,3,16.55,NaN,NaN,90,FALSE,mix and match allowed
4,PR2025091407,BOGO,Buy 1 Get 1,2025-09-14,2025-09-20,category,Meat and Seafood,1,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,80,FALSE,applies to category; singles only
5,PR2025091405,BOGO,Buy 1 Get 1,2025-09-14,2025-09-20,category,Deli,ALL,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,80,FALSE,applies to category; singles only
6,PR2025091411,BOGO,Buy 1 Get 1,2025-09-14,2025-09-20,category,Condiments/Sauces/Spices,ALL,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,80,FALSE,applies to category; singles only
7,PR2025091413,BOGO,Buy 1 Get 1,2025-09-14,2025-09-20,category,Alcohol,ALL,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,80,FALSE,applies to category; singles only
8,PR2025091410,COUPON,$1.00 Off (coupon),2025-09-14,2025-09-20,category,Bakery,2,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70,TRUE,coupon applies at checkout
9,PR2025091412,COUPON,$3.00 Off (coupon),2025-09-14,2025-09-20,category,Dairy & Eggs,2,3.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.00,70,TRUE,coupon applies at checkout


In [10]:

# Quick summary by promo_type / scope_type

if not current.empty:
    group_cols = [c for c in ["promo_type","scope_type","store_scope"] if c in current.columns]
    if group_cols:
        summary = (current.groupby(group_cols, dropna=False).size()
                   .reset_index(name="count")
                   .sort_values(["promo_type","count"] if "promo_type" in group_cols else ["count"],
                                ascending=[True, False] if "promo_type" in group_cols else [False]))
        display(summary)


,promo_type,scope_type,store_scope,count
1,BOGO,category,ALL,3
0,BOGO,category,1,1
4,BUNDLE,category,ALL,2
2,BUNDLE,category,2,1
3,BUNDLE,category,3,1
5,COUPON,category,2,2
6,COUPON,category,3,1
7,COUPON,category,ALL,1
9,MANAGER_SPECIAL,category,ALL,2
8,MANAGER_SPECIAL,category,1,1
